# 09. Benchmark: recall vs files scanned

Charts how much of the exact answer IVF recovers against how many files it has to open, from the saved benchmark results.

## What to look at

- The **chart**: each dot is an `nprobe` value. Up and to the left is better (high recall, few files).
- The **table** is read from `benchmarks/ivf_benchmark.json`; nothing here is typed in.
- Latency gains can be small on a laptop-sized table: Spark has a fixed cost per query. The files-scanned column is the clearer signal.

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
# The benchmark takes several minutes (50 queries x 5 methods x repeats), so by default we only READ the saved results.
# Set RUN_BENCHMARK = True to re-measure (this overwrites benchmarks/ivf_benchmark.*).
RUN_BENCHMARK = os.environ.get("NB_RUN_BENCHMARK") == "1"
if RUN_BENCHMARK:
    run_script("09_benchmark.py")
else:
    print("reading saved results from benchmarks/ivf_benchmark.json")

In [ ]:
data = json.loads((BENCH_DIR / "ivf_benchmark.json").read_text())
res = pd.DataFrame(data["results"])
print(f'{data["queries"]} queries | {data["gold_rows"]} vectors in {data["gold_data_files"]} data files | k={data["ivf_k"]}')
res[["config", "mean_ms", "p95_ms", "latency_speedup_vs_brute", "mean_files_scanned", "files_scanned_pct_of_brute",
     "mean_rows_scanned", "recall_at_10", "recall_at_10_tie_aware"]]

In [ ]:
import matplotlib.pyplot as plt
ivf = res[res.nprobe.notna()]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ivf.mean_files_scanned, ivf.recall_at_10, "o-", color="#2f6fed", label="IVF")
for _, r in ivf.iterrows():
    ax[0].annotate(f"nprobe={int(r.nprobe)}", (r.mean_files_scanned, r.recall_at_10), textcoords="offset points", xytext=(6, -12))
brute = res[res.nprobe.isna()].iloc[0]
ax[0].scatter([brute.mean_files_scanned], [brute.recall_at_10], color="#d9822b", zorder=3, label="brute force")
ax[0].set_xlabel("data files scanned (mean per query)"); ax[0].set_ylabel("recall@10"); ax[0].set_ylim(0, 1.05)
ax[0].set_title("Recall vs files scanned"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].bar(res.config, res.mean_ms, color=["#d9822b"] + ["#2f6fed"] * (len(res) - 1))
ax[1].set_ylabel("mean latency (ms)"); ax[1].set_title("Latency"); ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# Cleanup. Skip this cell while you are showing the Spark UI (http://localhost:4040): it closes with the session.
stop_spark()

## Open MinIO now

http://localhost:9001 (login `minioadmin` / `minioadmin`), bucket **warehouse**.

Nothing new: the benchmark only reads. To connect the chart to the files, open `gold/review_embeddings/data/` and note how many Parquet files there are in total: that is the 'brute force' point on the x-axis.